In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import torch
import h5py

import sys
sys.path.insert(0, "../../src")

import os
import torch.nn as nn
import torch.distributed as dist

from juart.dl.model.dc import DataConsistency
from juart.dl.model.regularizer import Regularizer
from tqdm import tqdm

from juart.dl.loss.loss import JointLoss
from juart.dl.model.unrollnet import LookaheadModel
from juart.dl.utils.dist import GradientAccumulator

from torch.utils.checkpoint import checkpoint
from juart.vis.interactive import InteractiveMultiPlotter3D
from juart.conopt.functional.fourier import nonuniform_fourier_transform_adjoint, nonuniform_fourier_transform_forward, fourier_transform_forward, fourier_transform_adjoint

In [ ]:
dist.init_process_group(
    backend="gloo", init_method="tcp://127.0.0.1:55832", world_size=1, rank=0
)

In [ ]:
def setup_DC(
    data: dict,
    shape: tuple[int],
    axes: tuple[int] = (1,2,3),
    device="cpu",
    verbose=True,
    niter: int = 10
):
    dc_block = DataConsistency(
        shape,
        axes = (1,2,3),
        device=device,
        verbose = True,
        niter = niter
    )

    dc_block.init(
        data["images_regridded"],
        data["kspace_trajectory"],
        sensitivity_maps=data["sensitivity_maps"],
        kspace_mask = data["kspace_mask_source"]
    )

    return dc_block

In [ ]:
def setup_RES(
    shape: tuple[int],
    features: int = 32,
    num_of_resblocks: int = 15,
    device: str = "cpu",
    kernel_size: tuple[int] = (3,3,3),
    Checkpoints: bool = True
):

    resnet = Regularizer(
        shape = shape,
        features=features,
        num_of_resblocks=num_of_resblocks,
        device=device,
        regularizer = 'ResNet',
        kernel_size=kernel_size,
        Checkpoints = Checkpoints)

    return resnet

In [ ]:
class UnrolledNet(nn.Module):
    def __init__(
        self,
        shape,
        CG_Iter=10,
        num_unroll_blocks=10,
        num_of_resblocks=15,
        scale_factor: int = 0,
        features=32,
        activation="ReLU",
        filter_name: str = None,
        filter_radius: int = 0,
        lamda_start=0.05,
        phase_normalization=False,
        disable_progress_bar=False,
        pad_to: int = 0,
        timing_level=0,
        validation_level=0,
        kernel_size: tuple[int] = (3, 3),
        regularizer="ResNet",
        device=None,
        Checkpoints:bool = False,
        dtype=torch.complex64,
    ):
        """
        Initializes an UnrollNet as a neural Network existing out of a regularizer
        and a data consistency layer.

        Parameters
        ----------
        shape : torch.Tensor, shape (nX, nY, nZ, nTI, nTE)
            Shape of the image data.
        CG_Iter : int, optional
            Number of the iterations in the conjugate gradient (Data Consistency) term
        num_unroll_blocks : int, optional
            Number of iterations in the loop of data consistency term and regularization
            term (default is 10).
        num_of_res_blocks : int, optional
            Number of ResNetBlocks that should be added to the second layer of the
            ResNet (default is 15).
        features : int, optional
            Number of the features of the neural network (default is 128).
        activation: str, optional
            defines the kind of activation function (default is "ReLu")
        phase_normalization: bool, optional
            normalizes the signals phase (default is False).
        disable_progress_bar: bool, optional
            Disable the progress bar output (default is False).
        kernel_size: Tuple[int], optional
            changes the size of the kernel used in the convolutional layers and  its length
            decides whether all operations should be 2D or 3D. (default is (3,3))
        regularizer: str, optional
            decides which regularizer should be used. For now there are ResNet and UNet
            (default is ResNet).
        pad_to: int, optional
            provides the ability to pad the input image to the shape (pad_to,pad_to,1) or
            (pad_to,pad_to,pad_to) depending on the length of the kernel_size. Originally
            used for the UNet and its dependency on the shape of 2^n. If pad_to = 0 and
            UNet is used than the shape will be padded to the next 2^n shape.
            (default is 0)
        device : str, optional
            Device on which to perform the computation
            (default is None, which uses the current device).
            It is also possible to give a list of strings. The first
            item is the DataConsistency device and the second one is
            the device used for the regularizer.
        Checkpoints: bool, optional
            If true then checkpoints will be added in the regularizer, providing lower memory
            usage to the cost of higher computing time (default is False).

        NOTE: This function is under development and may not be fully functional yet.
        """
        super().__init__()

        axis = ([n for n in range(1, len(kernel_size)+1, 1)])
        self.pad_to = pad_to
        self.net_structure = regularizer
        self.kernel_size = kernel_size
        self.phase_normalization = phase_normalization
        self.num_unroll_blocks = num_unroll_blocks
        self.disable_progress_bar = disable_progress_bar
        self.timing_level = timing_level
        self.validation_level = validation_level
        self.device = device

        nX, nY, nZ, nTI, nTE = shape

        self.regularizer = setup_RES(
            shape,
            features,
            num_of_resblocks,
            self.device,
            self.kernel_size,
            Checkpoints
        )

        self.dc = DataConsistency(
            shape,
            niter=CG_Iter,
            lamda_start=lamda_start,
            timing_level=timing_level - 1,
            validation_level=validation_level - 1,
            axes=axis,
            device=device,
            dtype=dtype,
        )

    def forward(
        self,
        images_regridded: torch.Tensor,
        kspace_trajectory: torch.Tensor,
        kspace_mask: torch.Tensor = None,
        sensitivity_maps: torch.Tensor = None,
    ) -> torch.Tensor:

        if self.phase_normalization:
            images_phase = torch.exp(1j * torch.angle(images_regridded[..., 0, 0]))
            images_regridded = images_regridded / images_phase[..., None, None]
            sensitivity_maps = sensitivity_maps * images_phase[None, :, :]

        self.dc.init(
            images_regridded,
            kspace_trajectory,
            kspace_mask=kspace_mask,
            sensitivity_maps=sensitivity_maps,
        )

        image = images_regridded.clone().detach()

        for _ in tqdm(range(self.num_unroll_blocks), disable=self.disable_progress_bar):
            image = checkpoint(self.regularizer, image, use_reentrant=False)
            image = checkpoint(self.dc, image, use_reentrant=False)

        if self.phase_normalization:
            image = image * images_phase[..., None, None]

        return image

In [ ]:
data_path = "/home/jovyan/datasets/kooshball_GRE_1000spk_preproc.h5"
with h5py.File(data_path, "r") as f:

    k = torch.from_numpy(f['traj'][:])[...,None,None]
    k = k.reshape(k.shape[0],k.shape[1]*k.shape[2],k.shape[3],k.shape[4])

    C = torch.from_numpy(f['sens_maps'][:])
    d = torch.from_numpy(f['data_comp'][:])[...,None,None]
    d = d.reshape(d.shape[0],d.shape[1]*d.shape[2],d.shape[3],d.shape[4])

    #normalization
    k /= (2*k.max())
    k = k.view(*k.shape, *([1]*(4-k.dim())))

    d /= d.abs().max()
    d = d.view(*d.shape, *([1]*(4-d.dim())))

    C /= C.abs().max()

    print(f"Coilsensitivity shape {C.shape}")
    print(f"Trajectory shape {k.shape}")
    print(f"Signal shape {d.shape}")

In [ ]:
kspace_mask_source = torch.randint(0,2,(1, k.shape[1], 1,1))
kspace_mask_target = 1 - kspace_mask_source

AHd = nonuniform_fourier_transform_adjoint(k,d,(128,128,128))
AHd = torch.sum(torch.conj(C[...,None,None]) * AHd, dim=0)

In [ ]:
data = {
    "images_regridded": AHd,
    "kspace_trajectory": k,
    "sensitivity_maps": C,
    "kspace_mask_source": kspace_mask_source,
    "kspace_mask_target": kspace_mask_target,
    "kspace_data": d,
}

In [ ]:
device = "cuda:2"
shape = (128,128,128,1,1)
dtype = torch.complex64

model = UnrolledNet(
    shape,
    CG_Iter=50,
    num_unroll_blocks=5,
    num_of_resblocks=15,
    features=32,
    kernel_size = (3,3,3),
    regularizer="ResNet",
    device=device,
    Checkpoints=True
)

In [ ]:
loss_fn = JointLoss(
    kernel_size=(3, 3, 3),
    weights_kspace_loss=(0.5, 0.5),
    weights_ispace_loss=(0.0, 0.0),
    weights_wavelet_loss=(0.0, 0.0),
    weights_hankel_loss=(0.0, 0.0),
    weights_casorati_loss=(0.0, 0.0),
    normalized_loss=True,
    timing_level=0,
    validation_level=0,
    device=device,
    dtype = dtype
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001,
    betas=[0.9, 0.999],
    eps=1.0e-8,
    weight_decay=0.0
)

accumulator = GradientAccumulator(
    model,
    accumulation_steps=1,
    max_norm=1.0,
    normalized_gradient=False
)

averaged_model = LookaheadModel(
    model,
    alpha=0.5,
    k=5
)

In [ ]:
losses = []
model.train()

with torch.autograd.graph.save_on_cpu():

    images_regridded = data["images_regridded"].to(device)
    kspace_trajectory = data["kspace_trajectory"].to(device)
    kspace_data = data["kspace_data"].to(device)
    kspace_mask_source = data["kspace_mask_source"].to(device)
    kspace_mask_target = data["kspace_mask_target"].to(device)
    sensitivity_maps = data["sensitivity_maps"].to(device)
    print(f"Rank {dist.get_rank()} - reading data done -> model initialization")

    images_reconstructed = model(
        images_regridded,
        kspace_trajectory,
        kspace_mask=kspace_mask_source,
        sensitivity_maps=sensitivity_maps,
    )
    
    print(f"Rank {dist.get_rank()} - model initialization done -> loss fn initialization")
    # Loss

    x = images_reconstructed.to(device)
    kspace_trajectory = kspace_trajectory.to(device)
    kspace_data = kspace_data.to(device)
    kspace_mask = kspace_mask_target.to(device)
    sensitivity_maps = sensitivity_maps.to(device)

    x = 0.5 * (
        images_reconstructed[None, ...] * sensitivity_maps[..., None, None]
    )

    kspace_data_reconstructed = nonuniform_fourier_transform_forward(
        kspace_trajectory,
        x,
    )

    kspace_data_reconstructed = kspace_data_reconstructed * kspace_mask

    kspace_reference = kspace_data * kspace_mask

    print(
        "[KSpaceLoss]",
        x.shape,
        x_reference.shape,
    )

    loss = torch.linalg.vector_norm(
        kspace_data_reconstructed - kspace_reference,
        ord=1,
        keepdim=True,
    )

    reference_loss = torch.linalg.vector_norm(
        kspace_reference,
        ord=1,
        keepdim=True,
    )

    loss = loss / reference_loss
    print(loss.shape)

    loss = torch.mean(loss)

    print(f"Rank {dist.get_rank()} - loss fn initialization done -> compute backward pass")
    dist.barrier() 
    # Backpropagation
    print(type(loss))
    print(loss)
    loss.backward()
    print(f"Rank {dist.get_rank()} - compute backward pass done -> compute accumulator")
    dist.barrier() 
    # Accumulate gradients
    accumulator.accumulate()

    losses.append(loss.item())

    accumulator.apply()

    optimizer.step()
    optimizer.zero_grad()

In [ ]:
    loss = torch.linalg.vector_norm(
        x - x_reference,
        ord=1,
        keepdim=True,
    )

    reference_loss = torch.linalg.vector_norm(
        x_reference,
        ord=1,
        keepdim=True,
    )